# Week 7: Document Question Answering System using RAG

## Retrieval-Augmented Generation

### Objective
Build a Document Question Answering System using Retrieval-Augmented Generation (RAG). The system processes a custom PDF, creates semantic embeddings, retrieves relevant document chunks for a question, reranks them, and generates an answer using the retrieved context.

### RAG Pipeline
**Document → Text Extraction → Chunking → Embeddings → FAISS Vector Store → Query → Retrieval → Reranking → Context → Language Model → Answer**

## 1. Introduction

Retrieval-Augmented Generation (RAG) combines information retrieval with language generation. Instead of relying only on a language model's internal knowledge, a RAG system retrieves relevant information from a custom document and uses it as context for generating an answer.

This approach is useful for question answering over private, domain-specific, and custom documents.

In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.8 MB/s eta 0:00:00


In [2]:
import os
import re
import fitz
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Custom PDF

The custom document used for this project is a PDF containing practical notes on Data Science and Deep Learning.

In [3]:
pdf_candidates = [f for f in os.listdir("/content") if f.lower().endswith(".pdf")]

print("PDF files found:")
for file in pdf_candidates:
    print("-", file)

if not pdf_candidates:
    raise FileNotFoundError("No PDF was found in /content. Please upload the project PDF.")

preferred_name = "Data_Science_Deep_Learning_Notes_for_RAG.pdf"
pdf_path = os.path.join("/content", preferred_name if preferred_name in pdf_candidates else pdf_candidates[0])

print("\nUsing document:", pdf_path)

PDF files found:
- Data_Science_Deep_Learning_Notes_for_RAG.pdf

Using document: /content/Data_Science_Deep_Learning_Notes_for_RAG.pdf


In [4]:
def extract_text_from_pdf(pdf_path):
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            page_text = page.get_text("text")
            if page_text.strip():
                pages.append(f"[Page {page_number}]\n{page_text.strip()}")
    return "\n\n".join(pages)

pdf_text = extract_text_from_pdf(pdf_path)

print("Characters extracted:", len(pdf_text))
print("Words extracted:", len(pdf_text.split()))
print("\nFirst 2000 characters:\n")
print(pdf_text[:2000])

Characters extracted: 9458
Words extracted: 1361

First 2000 characters:

[Page 1]
Data Science and Deep Learning — Practical
Notes
A compact technical reference document designed for demonstrating document question answering with
a Retrieval-Augmented Generation (RAG) system.
1. Introduction to Data Science
Data Science combines statistics, programming, machine learning, and domain knowledge to extract
useful information from data. A typical data science workflow includes problem definition, data collection,
data cleaning, exploratory data analysis, feature engineering, model development, evaluation, and
deployment. The quality of the final model depends heavily on the quality and relevance of the data used
during these stages.
2. Machine Learning
Machine learning allows a computer system to learn patterns from data and make predictions or
decisions without explicitly programming every rule. Supervised learning uses labeled examples, while
unsupervised learning works with data that do

## 3. Document Preprocessing and Text Chunking

Large documents are divided into smaller overlapping chunks so the retrieval system can identify specific relevant sections.

In [5]:
def split_text(text, chunk_size=900, overlap=150):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            boundary = max(
                text.rfind(". ", start, end),
                text.rfind("? ", start, end),
                text.rfind("! ", start, end)
            )
            if boundary > start + int(chunk_size * 0.6):
                end = boundary + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks

chunks = split_text(pdf_text)

print("Number of chunks:", len(chunks))
print("Average chunk length:", round(np.mean([len(c) for c in chunks]), 2))

for i, chunk in enumerate(chunks[:3], 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Number of chunks: 14
Average chunk length: 814.5

--- Chunk 1 ---
[Page 1] Data Science and Deep Learning — Practical Notes A compact technical reference document designed for demonstrating document question answering with a Retrieval-Augmented Generation (RAG) system. 1. Introduction to Data Science Data Science combines statistics, programming, machine learning, and domain knowledge to extract useful information from data. A typical data science workflow includes problem definition, data collection, data cleaning, exploratory data analysis, feature engineering, model development, evaluation, and deployment. The quality of the final model depends heavily on the quality and relevance of the data used during these stages. 2. Machine Learning Machine learning allows a computer system to learn patterns from data and make predictions or decisions without explicitly programming every rule.

--- Chunk 2 ---
ing Machine learning allows a computer system to learn patterns from data and make pr

## 4. Generate Text Embeddings

The `all-MiniLM-L6-v2` Sentence Transformer model converts document chunks and user questions into numerical vectors that capture semantic meaning.

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [7]:
document_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", document_embeddings.shape)
print("Embedding dimension:", document_embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (14, 384)
Embedding dimension: 384


## 5. Create the FAISS Vector Store

FAISS is used as a local vector store. Normalized vectors allow inner product search to act as cosine similarity search.

In [8]:
document_embeddings = document_embeddings.astype("float32")
faiss.normalize_L2(document_embeddings)

embedding_dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dimension)
index.add(document_embeddings)

print("FAISS index created successfully.")
print("Number of vectors stored:", index.ntotal)
print("Vector dimension:", embedding_dimension)

FAISS index created successfully.
Number of vectors stored: 14
Vector dimension: 384


## 6. Semantic Retrieval

When a user asks a question, the question is converted into an embedding and the vector index searches for the most similar document chunks.

In [9]:
def retrieve_documents(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, top_k)

    retrieved_documents = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            retrieved_documents.append({
                "text": chunks[int(idx)],
                "score": float(score),
                "chunk_index": int(idx)
            })
    return retrieved_documents

In [10]:
test_question = "What is Retrieval-Augmented Generation?"
retrieved_documents = retrieve_documents(test_question, top_k=3)

print("Question:", test_question)
for i, document in enumerate(retrieved_documents, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Similarity score:", round(document["score"], 4))
    print(document["text"])

Question: What is Retrieval-Augmented Generation?

--- Retrieved Chunk 1 ---
Similarity score: 0.5115
is not identical. 14. Retrieval-Augmented Generation Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation. A document is first loaded and divided into smaller chunks. Each chunk is converted into an embedding and stored in a vector index. When a user asks a question, the question is embedded and compared with document embeddings. The most relevant chunks are retrieved and supplied as context to a language model, which generates an answer based on that context. RAG is useful for question answering over custom documents because the system can retrieve information from the source instead of relying only on a model's internal knowledge. 15. Vector Databases and Similarity Search A vector store maintains vector representations and supports similarity search. FAISS is a library for efficient similarity search and clustering of dense vectors.

--- Re

## 7. Document Reranking

The initial vector search retrieves potentially relevant chunks. A Cross-Encoder reranks these chunks according to the relationship between the question and each chunk.

In [11]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranking model loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranking model loaded successfully.


In [12]:
def rerank_documents(query, retrieved_documents, top_k=3):
    if not retrieved_documents:
        return []

    pairs = [[query, document["text"]] for document in retrieved_documents]
    scores = reranker.predict(pairs)

    ranked_documents = []
    for document, score in zip(retrieved_documents, scores):
        ranked_documents.append({
            "text": document["text"],
            "score": float(score),
            "chunk_index": document["chunk_index"]
        })

    ranked_documents.sort(key=lambda item: item["score"], reverse=True)
    return ranked_documents[:top_k]

In [13]:
question = "What are the main stages of a RAG pipeline?"
retrieved_documents = retrieve_documents(question, top_k=5)
reranked_documents = rerank_documents(question, retrieved_documents, top_k=3)

print("Question:", question)
for i, document in enumerate(reranked_documents, 1):
    print(f"\n--- Reranked Chunk {i} ---")
    print("Reranking score:", round(document["score"], 4))
    print(document["text"])

Question: What are the main stages of a RAG pipeline?

--- Reranked Chunk 1 ---
Reranking score: 4.0727
blem do LSTMs address in standard RNNs? 4. What is an autoencoder and where can it be used? 5. What is the difference between precision and recall? 6. What is overfitting and how can it be reduced? 7. What are embeddings used for in semantic search? 8. What are the main stages of a RAG pipeline? 9. Why is text chunking important in a RAG system? 10. How does FAISS help a document question-answering system?

--- Reranked Chunk 2 ---
Reranking score: 3.4456
e maintains vector representations and supports similarity search. FAISS is a library for efficient similarity search and clustering of dense vectors. Cosine similarity and inner product are commonly used to compare embeddings. A typical retrieval system selects the top-k chunks with the highest similarity to the query. The retrieved chunks can then be passed to a reranker or directly used as context for answer generation. 16. Pract

## 8. Load the Language Model

A lightweight FLAN-T5 model is used for the generation stage. The retrieved context and user question are supplied to the model.

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
language_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Language model loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Language model loaded successfully.


## 9. Complete RAG Question Answering Function

The function below combines retrieval, reranking, context construction, and answer generation.

In [18]:
def answer_question(question, retrieval_k=5, context_k=3):

    # Step 1: Retrieve relevant document chunks
    retrieved_documents = retrieve_documents(
        question,
        top_k=retrieval_k
    )

    # Step 2: Rerank the retrieved chunks
    reranked_documents = rerank_documents(
        question,
        retrieved_documents,
        top_k=context_k
    )

    # Step 3: Create context from the best chunks
    context = "\n\n".join(
        document["text"]
        for document in reranked_documents
    )

    # Step 4: Create the RAG prompt
    prompt = f"""
Answer the question using only the information provided in the context.

If the answer is not available in the context, say that the information
is not available in the provided document.

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 5: Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Step 6: Generate the answer using FLAN-T5
    outputs = language_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    # Step 7: Convert generated tokens back to text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": retrieved_documents,
        "reranked_documents": reranked_documents,
        "context": context
    }

## 10. Single Question Demonstration

In [19]:
question = "What is the purpose of text chunking?"
result = answer_question(question)

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(question)

print("\n" + "=" * 80)
print("RETRIEVED AND RERANKED CONTEXT")
print("=" * 80)

for i, document in enumerate(result["reranked_documents"], 1):
    print(f"\n--- Context {i} ---")
    print(document["text"])

print("\n" + "=" * 80)
print("GENERATED ANSWER")
print("=" * 80)
print(result["answer"])

QUESTION
What is the purpose of text chunking?

RETRIEVED AND RERANKED CONTEXT

--- Context 1 ---
e maintains vector representations and supports similarity search. FAISS is a library for efficient similarity search and clustering of dense vectors. Cosine similarity and inner product are commonly used to compare embeddings. A typical retrieval system selects the top-k chunks with the highest similarity to the query. The retrieved chunks can then be passed to a reranker or directly used as context for answer generation. 16. Practical RAG Workflow A practical document question-answering pipeline can be organized into seven stages: document ingestion, text chunking, embedding creation, vector storage, query processing, context retrieval, and answer generation. Good chunking is important because chunks that are too small may lose context, while chunks that are too large may reduce retrieval precision.

--- Context 2 ---
blem do LSTMs address in standard RNNs? 4. What is an autoencoder and 

## 11. Multiple Question Evaluation

The system is tested with questions covering different sections of the custom document.

In [20]:
questions = [
    "What is the purpose of an activation function in a neural network?",
    "How does a CNN process image data?",
    "What problem do LSTMs address in standard RNNs?",
    "What is an autoencoder and where can it be used?",
    "What is the difference between precision and recall?",
    "What is overfitting and how can it be reduced?",
    "What are embeddings used for in semantic search?",
    "What are the main stages of a RAG pipeline?",
    "Why is text chunking important in a RAG system?",
    "How does FAISS help a document question-answering system?"
]

results = []

for question in questions:
    result = answer_question(question)
    results.append(result)

    print("=" * 80)
    print("QUESTION")
    print("=" * 80)
    print(question)
    print("\nGENERATED ANSWER")
    print("-" * 80)
    print(result["answer"])
    print()

QUESTION
What is the purpose of an activation function in a neural network?

GENERATED ANSWER
--------------------------------------------------------------------------------
1. Activation functions introduce non-linearity into neural networks 2. Using compact representations and can reconstruct or denoise inputs 3. Using a neural network 4. RNNs, LSTMs, and GRUs 5. Using semantic retrieval and language generation 6. ing Machine learning allows a computer system to learn patterns from data and make predictions or decisions without explicitly programming every rule

QUESTION
How does a CNN process image data?

GENERATED ANSWER
--------------------------------------------------------------------------------
1. learn compact representations 2. reconstruct or denoise inputs 3. combines semantic retrieval with language generation 4. RAG 5. reconstruct or denoise inputs 6. RAG 7. Recurrent Neural Networks Recurrent Neural Networks

QUESTION
What problem do LSTMs address in standard RNNs?

GE

## 12. Retrieval Inspection

Inspecting retrieved context helps evaluate whether the information supplied to the language model is relevant to the question.

In [21]:
for result in results[:5]:
    print("=" * 80)
    print("QUESTION:", result["question"])
    print("=" * 80)

    for i, document in enumerate(result["reranked_documents"], 1):
        print(f"\nContext {i} | Reranking Score: {document['score']:.4f}")
        print(document["text"][:500])
        print()

QUESTION: What is the purpose of an activation function in a neural network?

Context 1 | Reranking Score: 5.5144
ayers, and an output layer. Each connection has a weight, and neurons commonly apply an activation function to a weighted combination of their inputs. During training, the network adjusts its weights to reduce a loss function. 4. Activation Functions Activation functions introduce non-linearity into neural networks. ReLU, or Rectified Linear Unit, outputs zero for negative inputs and the input itself for positive inputs. Sigmoid maps values into a range between zero and one and is often used for


Context 2 | Reranking Score: 3.5774
e based on the requirements of the application. 18. Key Takeaways Data science involves a complete workflow from data preparation to model deployment. Deep learning models learn representations through multiple layers of trainable parameters. CNNs are effective for spatial data, while RNNs, LSTMs, and GRUs are useful for sequential data. Autoenc

## 13. RAG System Architecture

**Custom PDF**
↓  
**Document Ingestion**
↓  
**Text Chunking**
↓  
**Embedding Creation**
↓  
**FAISS Vector Store**
↓  
**User Query**
↓  
**Query Embedding**
↓  
**Similarity Retrieval**
↓  
**Document Reranking**
↓  
**Relevant Context**
↓  
**FLAN-T5 Language Model**
↓  
**Generated Answer**

## 14. Results and Observations

The RAG system processes a custom PDF and uses its content as the knowledge source for question answering.

### Observations

1. PDF text was extracted successfully.
2. The document was divided into smaller overlapping chunks.
3. Sentence Transformer embeddings were generated.
4. FAISS provided local vector similarity search.
5. Relevant chunks were retrieved for user questions.
6. A Cross-Encoder reranked the retrieved chunks.
7. The highest-ranked chunks were used as generation context.
8. FLAN-T5 generated answers using the retrieved context.

The actual answers and similarity scores displayed in this notebook are produced during execution.

## 15. Advantages

- Works with custom documents.
- Does not require fine-tuning the language model.
- Uses semantic retrieval.
- Can run with open-source models in Google Colab.
- Retrieved context can be inspected.
- The source PDF can be replaced with another suitable document.

## 16. Limitations

- Answer quality depends on document extraction and retrieval quality.
- Small language models may provide less detailed answers.
- Poor chunk sizes can reduce retrieval performance.
- FAISS in this project is a local, non-persistent vector store.
- Generated answers should be checked against retrieved context for important applications.

## 17. Possible Improvements

- Better chunking strategies.
- Different or stronger embedding models.
- Hybrid keyword and vector search.
- More advanced reranking.
- Larger language models.
- A Streamlit or web interface.
- A persistent vector database such as Pinecone.
- A larger evaluation question set.

## 18. Conclusion

This project implemented a Document Question Answering System using Retrieval-Augmented Generation.

The system loads a custom PDF, extracts its text, creates overlapping chunks, generates embeddings, stores the embeddings in a FAISS vector index, retrieves relevant chunks, reranks them, and generates answers using a language model.

The project demonstrates how retrieval and generation can be combined to answer questions using custom documents.

## 19. Viva Questions and Answers

### Q1. What is RAG?
RAG stands for Retrieval-Augmented Generation. It retrieves relevant external information and provides it as context to a language model before generating an answer.

### Q2. Why is document chunking required?
Chunking divides a large document into smaller sections so relevant information can be retrieved more effectively.

### Q3. What are embeddings?
Embeddings are numerical vector representations that capture semantic information.

### Q4. Why is FAISS used?
FAISS provides efficient similarity search over vector embeddings and works as a local vector store.

### Q5. What is reranking?
Reranking reorders initially retrieved documents according to their relevance to the query.

### Q6. What is the role of the language model?
It uses the retrieved context and question to generate the final natural-language answer.

### Q7. What is the difference between RAG and fine-tuning?
RAG retrieves information at query time, while fine-tuning changes model parameters through additional training.

### Q8. What happens if the answer is not in the document?
A well-designed RAG system should indicate that the information is unavailable rather than inventing unsupported information.

### Q9. Why are embeddings useful?
They allow semantic similarity search, so relevant text can be found even when wording differs.

### Q10. What are possible RAG improvements?
Better chunking, stronger embeddings, hybrid search, improved reranking, larger language models, and persistent vector databases.